# Causal attention: can the future change the past?

Chapter 4 · Session 1 of 3 · CPU, tiny float64 tensors.

Work through one checkpoint at a time. Record your prediction before opening its reference solution. Every exercise has an adjacent runnable answer and explanation. You can run all reference cells unchanged to check the environment, but that does not replace the discussion.

We will trace Q/K/V, implement attention, break masking in two ways, and test whether changing the future changes the past. No training is required.

## 1. Prediction checkpoint: two different futures

Imagine the token positions `The | animal | crossed | river` and `The | animal | crossed | road`. These are illustrative token labels; we are not claiming a particular tokenizer splits this way.

The first three input vectors are identical. Only the fourth changes. With fixed weights and dropout disabled, should the first three attention outputs change? What about the fourth?

Write a qualitative prediction. No arithmetic is required.

In [ ]:
prediction_prefix = ""  # Your explanation; this cell is intentionally optional.
print(prediction_prefix or "Prediction not recorded yet.")

### Reference solution and explanation

The first three outputs must remain unchanged. Each can use only its own position and earlier positions. The fourth output may change because its own query, key, and value can change. We assume the input representations themselves respect the causal boundary: a mask cannot repair future information already mixed into X by an earlier faulty layer.

An output is a feature vector. It becomes next-token logits only after the downstream layers and output head.

In [ ]:
from pathlib import Path
import sys
import math
import torch
import torch.nn.functional as F

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/dongxi_llms/causal_attention_lab.py').exists())
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
from dongxi_llms.causal_attention_lab import attention_trace, teaching_inputs

torch.set_printoptions(precision=5, sci_mode=False)
x, wq, wk, wv = teaching_inputs()
q, k, v = x @ wq, x @ wk, x @ wv
for name, tensor in [('X', x), ('W_Q', wq), ('Q', q), ('K', k), ('V', v)]:
    print(name, tuple(tensor.shape), '\n', tensor)
print('Environment:', torch.__version__, '| device:', x.device, '| dtype:', x.dtype)

Here X has shape `[T,D]=[4,3]`; the three projections have shape `[D,2]`. Q and K have shape `[T,d_k]=[4,2]`; V has shape `[T,d_v]=[4,2]`. The two feature widths happen to match here but need not.

The fixed numbers are untrained teaching features. Their coordinates have no assigned linguistic meaning.

## 2. Implementation checkpoint: one query meets all keys

Implement the five steps below. The score matrix is `[T,T]`: row i is the receiving query, column j is a candidate source. Normalize over the last dimension (source positions).

`allowed[i,j]` should be true exactly when `j <= i`, including the diagonal. Use `masked_fill` rather than multiplying by negative infinity: `0 * -inf` produces NaN.

In [ ]:
def learner_attention(q, k, v):
    # Replace the placeholders, then set run_my_implementation=True.
    scores = ...       # Q @ K.T divided by sqrt(d_k)
    allowed = ...      # Boolean lower triangle, diagonal included
    masked = ...       # Forbidden scores replaced with -torch.inf
    weights = ...      # Softmax over keys
    output = ...       # Weighted value mixture
    return weights, output

run_my_implementation = False
if run_my_implementation:
    my_weights, my_output = learner_attention(q, k, v)


### Reference solution and explanation

Each query-key dot product scores compatibility. Scaling controls the score spread. The mask removes forbidden candidates before the shared softmax denominator is formed. Multiplying the resulting weights by V constructs a new feature vector.

In [ ]:
scores = (q @ k.T) / math.sqrt(q.shape[-1])
allowed = torch.ones_like(scores, dtype=torch.bool).tril()
masked = scores.masked_fill(~allowed, -torch.inf)
a = torch.softmax(masked, dim=-1)
o = a @ v
for name, tensor in [('scores', scores), ('allowed', allowed),
                     ('masked scores', masked), ('attention weights', a), ('output', o)]:
    print(name, '\n', tensor)
if run_my_implementation:
    torch.testing.assert_close(my_weights, a)
    torch.testing.assert_close(my_output, o)

## 3. Prediction checkpoint: what would count as evidence?

A row that sums to one looks like a probability distribution. Is that sufficient evidence of causal correctness? Which further properties should we inspect?

In [ ]:
prediction_checks = ""

### Reference solution and explanation

Check nonnegative weights, row sum one, zero future weights, and output equality with an explicit sum over allowed values. Then perform an intervention: change only future inputs and compare earlier outputs. Finally compare with an independent implementation. Row sums alone cannot detect future leakage.

In [ ]:
torch.testing.assert_close(a.sum(-1), torch.ones(4, dtype=x.dtype))
assert (a >= 0).all()
assert a[~allowed].count_nonzero() == 0
for i in range(x.shape[0]):
    explicit_mixture = (a[i, :i+1, None] * v[:i+1]).sum(dim=0)
    torch.testing.assert_close(o[i], explicit_mixture)
reference = F.scaled_dot_product_attention(
    q[None, None], k[None, None], v[None, None],
    is_causal=True, dropout_p=0.0)[0, 0]
torch.testing.assert_close(o, reference, atol=1e-12, rtol=1e-12)
print('All forward checks passed; reference max error:', (o-reference).abs().max().item())

## 4. Prediction checkpoint: can a zero future weight still leak information?

For one receiving position, keep two allowed scores at zero and vary a forbidden third score. Compare:

1. Replace its score with negative infinity before softmax.
2. Compute softmax across all scores, then zero its weight.
3. Replace its score with zero before softmax.

Which methods give zero future probability? Which preserve row sum one? Which still let the future affect an output? Predict first, then vary `future_score`.

In [ ]:
prediction_masking = ""
future_score = 10.0  # Try 0, 10, and -10 after predicting.

### Reference solution and explanation

Method 1 gives `[0.5,0.5,0]`. Method 2 gives zero future weight, but its score has already changed the denominator and reduced the allowed weights. Method 3 gives the forbidden candidate a numerator of `exp(0)=1`; its value still contributes.

Replacing a score with zero is different from adding zero to an allowed score. In an additive mask, zero means leave that score unchanged.

In [ ]:
s = torch.tensor([0., 0., future_score], dtype=torch.float64)
keep = torch.tensor([True, True, False])
correct = s.masked_fill(~keep, -torch.inf).softmax(-1)
post = s.softmax(-1).masked_fill(~keep, 0)
zero_score = s.masked_fill(~keep, 0).softmax(-1)
for name, weights in [('correct', correct), ('post-softmax zero', post), ('zero score', zero_score)]:
    print(name, '| weights:', weights, '| row sum:', weights.sum().item())

## 5. Prediction checkpoint: perform the future intervention

Return to the full Q/K/V computation. Keep the first three rows of X fixed and change only the last row. Does each implementation preserve the first three outputs?

Do not stop at inspecting future weights. Test the output itself.

In [ ]:
prediction_intervention = ""
future_vector = [8., -3., 4.]  # Change this after your first comparison.

### Reference solution and explanation

Correct masking preserves the first three outputs. Naive post-softmax zeroing can change their allowed weights through the future key. Zero-score replacement still lets the changed future value enter their mixtures. The fourth output is allowed to change in every method.

In [ ]:
changed = x.clone()
changed[-1] = torch.tensor(future_vector, dtype=x.dtype)
for mode in ['causal', 'post_softmax', 'zero_scores']:
    before = attention_trace(q, k, v, mode=mode)
    after = attention_trace(changed @ wq, changed @ wk, changed @ wv, mode=mode)
    errors = (before['output'] - after['output']).abs().max(dim=-1).values
    print(mode, '| maximum difference by position:', errors)
    if mode == 'causal':
        torch.testing.assert_close(before['output'][:-1], after['output'][:-1],
                                   atol=1e-12, rtol=1e-12)
print('Try another future_vector. Earlier causal outputs must still agree.')

## 6. Optional deeper question: can renormalizing repair late masking?

If we zero forbidden probabilities after softmax and then divide the remaining weights by their sum, would that recover causal attention? What numerical problem could remain?

In [ ]:
prediction_renormalize = ""

### Reference solution and explanation

In exact arithmetic, renormalization cancels the original denominator and recovers the distribution over allowed positions. This is a useful exception to the claim that masking must literally occur first.

In finite precision, a very large forbidden score can cause every allowed probability to underflow to zero before the repair. Renormalization then divides zero by zero. Masking scores before softmax prevents the forbidden maximum from causing that failure.

In [ ]:
for future in [10., 10000.]:
    s = torch.tensor([0., 0., future], dtype=torch.float64)
    late = s.softmax(-1).masked_fill(~keep, 0)
    repaired = late / late.sum()
    early = s.masked_fill(~keep, -torch.inf).softmax(-1)
    print('Future score:', future, '| repaired:', repaired, '| early:', early)
    if future == 10.:
        torch.testing.assert_close(repaired, early)
    else:
        assert torch.isnan(repaired).all()  # Expected failure demonstration
        assert torch.isfinite(early).all()

## 7. Evidence checkpoint and next session

What has this notebook established, and what remains to be checked before calling Day 4 complete?

In [ ]:
my_evidence_boundary = ""

### Reference solution and explanation

We verified a transparent forward computation against PyTorch on tiny tensors and demonstrated two masking failures by intervention. Exact arithmetic explains why prefix invariance should hold generally for this causal computation; the finite test alone is not a proof for every implementation.

We have not trained a language model, tested attention gradients, measured GPU serving speed, or verified KV-cache equivalence. Those are separate claims. Sessions 2 and 3 will trace backward credit and incremental decoding. Learner understanding is assessed through our discussion and your interpretations, not merely successful execution.

The mask transition is already recorded for animation in `ANIM-ATTN-001`; rendering belongs on the Mac Studio.